# 4. Ranking Model (LightGBM)

Mục tiêu:
1. Tải `labeled_sessions.parquet` (chứa các session, user_id, product_id và label 0/1).
2. Nối (Join) với `user_features.parquet` và `item_features.parquet` để tạo bộ dữ liệu huấn luyện dạng bảng (Tabular Data).
3. Huấn luyện mô hình Tree-based (LightGBM) để dự đoán CTR.
4. Đánh giá mô hình bằng NDCG và MRR.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
import os
import warnings
warnings.filterwarnings('ignore')

### 1. Load & Join Data

In [ ]:
data_dir = '../data/'

# 1. Load labeled interactions
df_interactions = pd.read_parquet(os.path.join(data_dir, 'sessions/labeled_sessions.parquet'))

# 2. Load feature stores
user_features = pd.read_parquet(os.path.join(data_dir, 'feature_store/user_features.parquet'))
item_features = pd.read_parquet(os.path.join(data_dir, 'feature_store/item_features.parquet'))

# 3. Join
df = df_interactions.merge(user_features, on='user_id', how='left')
df = df.merge(item_features, on='product_id', how='left')

print(f"Joined Data Shape: {df.shape}")
df.head()

### 2. Prepare Train/Test Split

In [ ]:
# Giả lập chia Train/Test theo tỷ lệ 80/20 dựa trên User Session
features = [
    'user_total_interactions', 'user_total_sessions', 
    'item_total_interactions', 'item_unique_users', 'item_avg_price'
]
target = 'label'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

### 3. LightGBM Training

In [ ]:
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'binary',
    'metric': ['binary_logloss', 'auc'],
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1
    # 'device': 'gpu' # Uncomment if LightGBM GPU is compiled
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=100,
    valid_sets=[test_data]
)

# Đánh giá sơ bộ
preds = model.predict(X_test)
auc = roc_auc_score(y_test, preds)
print(f"Test AUC: {auc:.4f}")